# A3 — Facteurs latents

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| A — Recommandation de films | 3 sur 4 | 3 h 30 | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

La méthode des voisins améliore nettement les prévisions, mais elle reste une recette : il faut choisir $k$ et $\beta$, stocker une énorme matrice de similarités, et elle n'explique rien des goûts des spectateurs.

Au module M8, une matrice de températures se résumait par deux « profils types » et deux coefficients par station. Les goûts en matière de films pourraient se résumer de la même façon : quelques **facteurs cachés** (plutôt action ou plutôt intimiste, plutôt grand public ou plutôt exigeant…), un profil par film et un dosage par spectateur.

**La sous-question de cette étape : peut-on approcher la matrice des notes par une matrice de rang faible, alors que 95 % de ses cases sont inconnues ? Combien de facteurs faut-il, et lequel des outils fonctionne le mieux ?**

**Activité** : Au module M8, l'approximation de rang faible s'obtenait par la SVD d'une matrice **complète**. Quel problème pose une matrice presque vide ? Proposez une façon simple de la compléter avant la SVD, et dites ce qu'elle risque de fausser.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_nombre_facteurs = 20        # nombre de facteurs donnant la meilleure erreur de validation (modifiez)
pari_meilleur_outil = "SVD"      # "SVD" (matrice complétée par des zéros) ou "moindres carrés alternés" (modifiez)

## ③ Abstraire : une factorisation de la matrice des écarts

On cherche deux matrices de $k$ colonnes : $P$ (une ligne $\mathbf{p}_u$ par spectateur) et $Q$ (une ligne $\mathbf{q}_f$ par film), telles que l'écart au modèle des biais soit bien approché par leur produit :

$$
e_{uf} \approx \mathbf{p}_u \cdot \mathbf{q}_f
\qquad\text{donc}\qquad
\hat r_{uf} = \mu + b_u + b_f + \mathbf{p}_u \cdot \mathbf{q}_f .
$$

![Schéma : la matrice des écarts, partiellement connue, est approchée par le produit d'une matrice P de goûts et d'une matrice Q de profils ; on ajuste alternativement P puis Q par moindres carrés](../../../assets/schemas/schema-factorisation.svg)

**Outil 1 : la SVD d'une matrice complétée.** On remplace les cases inconnues par 0 (« pas d'écart au modèle des biais ») et l'on garde les $k$ premières couches de la SVD (module M8). Simple, mais les zéros ajoutés sont traités comme de vraies observations : la méthode « croit » que les spectateurs n'ont pas d'avis sur 95 % des films.

**Outil 2 : les moindres carrés alternés.** On ne tient compte **que des cases connues**, et l'on minimise

$$
\sum_{\text{notes connues}} \left(e_{uf} - \mathbf{p}_u \cdot \mathbf{q}_f\right)^2 + \lambda\left(\sum_u \lVert \mathbf{p}_u \rVert^2 + \sum_f \lVert \mathbf{q}_f \rVert^2\right).
$$

Si $Q$ est fixée, chaque $\mathbf{p}_u$ est la solution d'un petit problème de moindres carrés régularisés (module M6), à $k$ inconnues :

$$
\left(Q_u^{\mathsf{T}}Q_u + \lambda I\right)\mathbf{p}_u = Q_u^{\mathsf{T}}\mathbf{e}_u ,
$$

où $Q_u$ ne garde que les lignes des films notés par $u$. Puis on fixe $P$ et l'on calcule chaque $\mathbf{q}_f$ de la même façon, et l'on recommence. Chaque demi-étape fait baisser la quantité à minimiser.

**Activité** : Avec $k = 1$ et $\lambda = 0$, un spectateur a noté deux films de profils $q_1 = 2$ et $q_2 = 1$, avec des écarts $e_1 = 1$ et $e_2 = 0{,}2$. Calculez son goût $p$ par la formule. Vérifiez que c'est bien la valeur qui minimise $(1 - 2p)^2 + (0{,}2 - p)^2$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Préparation

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from films import charger_films, decouper, reqm, Index, ajuster_biais, predire_biais

notes, films, source = charger_films()
reste, test = decouper(notes, part_test=0.2, graine=0)      # le test reste sous scellés
app, val = decouper(reste, part_test=0.2, graine=1)
modele = ajuster_biais(app, lam=10)
mu, bu, bf = modele

idx = Index(app)                                             # cette fois, tous les films notés
R = idx.matrice(app)
connu = ~np.isnan(R)
base = mu + np.array([bu.get(u, 0) for u in idx.spectateurs])[:, None] + np.array([bf.get(f, 0) for f in idx.films])[None, :]
E = np.where(connu, R - base, 0.0)

lignes_val = idx.lignes(val["userId"])
colonnes_val = idx.colonnes(val["movieId"])
connu_val = colonnes_val >= 0
pred_biais_val = predire_biais(modele, val)

def evaluer(ecarts_predits):
    # REQM de validation quand on ajoute les écarts prédits au modèle des biais
    pred = pred_biais_val.copy()
    pred[connu_val] += ecarts_predits[lignes_val[connu_val], colonnes_val[connu_val]]
    return reqm(val["rating"], np.clip(pred, 0.5, 5))

reqm_biais = reqm(val["rating"], pred_biais_val)
print(f"Matrice {E.shape[0]} × {E.shape[1]} ; modèle des biais : {reqm_biais:.4f}")

### Outil 1 : la SVD de la matrice complétée par des zéros

In [ ]:
U, s, Vt = np.linalg.svd(E, full_matrices=False)
resultats_svd = {}
for k in [1, 2, 5, 10, 20, 40, 80]:
    resultats_svd[k] = evaluer(U[:, :k] @ np.diag(s[:k]) @ Vt[:k])
    print(f"SVD, k = {k:3d} : {resultats_svd[k]:.4f}")

### Outil 2 : les moindres carrés alternés

In [ ]:
def moindres_carres_alternes(E, connu, k=10, lam=10.0, iterations=15, graine=0, suivi=None):
    rng = np.random.default_rng(graine)
    P = rng.normal(0, 0.1, (E.shape[0], k))
    Q = rng.normal(0, 0.1, (E.shape[1], k))
    I = np.eye(k)
    for it in range(iterations):
        for u in range(E.shape[0]):                               # Q fixée : un petit système par spectateur
            j = connu[u]
            A = Q[j]
            P[u] = np.linalg.solve(A.T @ A + lam * I, A.T @ E[u, j])
        for f in range(E.shape[1]):                               # P fixée : un petit système par film
            i = connu[:, f]
            A = P[i]
            Q[f] = np.linalg.solve(A.T @ A + lam * I, A.T @ E[i, f])
        if suivi is not None:
            suivi.append((reqm(E[connu], (P @ Q.T)[connu]), evaluer(P @ Q.T)))
    return P, Q

suivi = []
debut = time.perf_counter()
P, Q = moindres_carres_alternes(E, connu, k=10, lam=10, iterations=20, suivi=suivi)
print(f"20 itérations en {time.perf_counter() - debut:.1f} s")

suivi = np.array(suivi)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(range(1, 21), suivi[:, 0], "o-", ms=3, color=LIOUAI["navy"], label="apprentissage (écarts)")
ax.plot(range(1, 21), suivi[:, 1], "s-", ms=3, color=LIOUAI["vert"], label="validation (notes)")
ax.set_xlabel("itération"); ax.set_ylabel("REQM"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Moindres carrés alternés, k = 10, λ = 10")
plt.tight_layout(); plt.show()

**Activité** : Au bout de combien d'itérations le gain sur la validation devient-il négligeable (moins de 0,001 étoile par itération) ? Pourquoi l'erreur d'apprentissage est-elle bien plus basse que celle de validation ?

Indiquez vos réponses et justifiez-les.

### Régler $k$ et $\lambda$

In [ ]:
grille = {}
debut = time.perf_counter()
for k in [2, 5, 10, 20]:
    for lam in [1, 3, 10, 30]:
        Pk, Qk = moindres_carres_alternes(E, connu, k=k, lam=lam, iterations=10)
        grille[(k, lam)] = evaluer(Pk @ Qk.T)
print(f"Grille calculée en {time.perf_counter() - debut:.0f} s")
tableau = pd.Series(grille).unstack().round(4)
tableau.index.name, tableau.columns.name = "k", "λ"
print(tableau.to_string())
k_opt, lam_opt = min(grille, key=grille.get)
print(f"\nMeilleur réglage : k = {k_opt}, λ = {lam_opt}, REQM {grille[(k_opt, lam_opt)]:.4f}")

### Que représentent les facteurs ?

Avec seulement deux facteurs, chaque film devient un point du plan. On regarde où se placent les genres.

In [ ]:
P2, Q2 = moindres_carres_alternes(E, connu, k=2, lam=10, iterations=15)
infos = films.set_index("movieId")
nb_notes = connu.sum(axis=0)
assez = nb_notes >= 30
genre_principal = [infos.loc[f, "genres"].split("|")[0] for f in idx.films]
a_montrer = ["Drama", "Comedy", "Action", "Horror", "Children", "Documentary"]

fig, ax = plt.subplots(figsize=(8, 6))
couleurs = [LIOUAI["navy"], LIOUAI["jaune"], LIOUAI["vert"], "#C0504D", LIOUAI["gris"], LIOUAI["vert_texte"]]
for g, c in zip(a_montrer, couleurs):
    m = assez & (np.array(genre_principal) == g)
    ax.scatter(Q2[m, 0], Q2[m, 1], s=14, color=c, label=g, alpha=0.8)
ax.axhline(0, color=LIOUAI["gris"], lw=0.6); ax.axvline(0, color=LIOUAI["gris"], lw=0.6)
ax.set_xlabel("facteur 1"); ax.set_ylabel("facteur 2"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("Les films (au moins 30 notes) dans le plan des deux facteurs")
plt.tight_layout(); plt.show()

for a in range(2):
    ordre = np.argsort(Q2[:, a] * assez)
    print(f"\nFacteur {a + 1}, extrémité positive :", "; ".join(infos.loc[idx.films[o], "genres"] for o in ordre[-4:]))
    ordre = np.argsort(np.where(assez, Q2[:, a], np.inf))
    print(f"Facteur {a + 1}, extrémité négative :", "; ".join(infos.loc[idx.films[o], "genres"] for o in ordre[:4]))

### Bilan comparatif

In [ ]:
debut = time.perf_counter()
P_opt, Q_opt = moindres_carres_alternes(E, connu, k=k_opt, lam=lam_opt, iterations=15)
temps_als = time.perf_counter() - debut
k_svd = min(resultats_svd, key=resultats_svd.get)
bilan = pd.DataFrame({
    "REQM validation": [reqm_biais, resultats_svd[k_svd], evaluer(P_opt @ Q_opt.T)],
    "nombres stockés": [len(bu) + len(bf) + 1, len(bu) + len(bf) + k_svd * sum(E.shape), len(bu) + len(bf) + k_opt * sum(E.shape)],
}, index=["biais", f"SVD des zéros (k = {k_svd})", f"moindres carrés alternés (k = {k_opt}, λ = {lam_opt})"])
print(bilan.round(4).to_string())
print(f"\nTemps des moindres carrés alternés : {temps_als:.1f} s")

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : {pari_nombre_facteurs} facteurs ; meilleur outil : {pari_meilleur_outil}")
print(f"Résultats : k = {k_opt} pour les moindres carrés alternés, k = {k_svd} pour la SVD")

**Activité** :

1. Quel outil retenez-vous ? Comparez-le aussi à la méthode des voisins de l'étape A2 (même paquet de validation).
2. Pourquoi la SVD de la matrice complétée par des zéros fait-elle moins bien ? Pourquoi son erreur remonte-t-elle quand $k$ augmente ?
3. Les deux facteurs ont-ils un sens ? Donnez-leur un nom provisoire, avec prudence.
4. Faut-il continuer à améliorer le modèle (plus de facteurs, d'autres réglages) ou passer à l'évaluation finale ? Sur quels chiffres fondez-vous la décision ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les biais figés.** On a ajusté les biais d'abord, puis les facteurs sur les écarts. On pourrait ajuster le tout ensemble. Qu'est-ce que cela changerait, a priori ?
- **Le hasard du départ.** Les moindres carrés alternés partent de matrices tirées au hasard. Relancez avec d'autres graines : les erreurs changent-elles ? Les facteurs eux-mêmes ?
- **Généraliser.** Écrivez une fonction `demi_etape(A, E, connu, lam)` qui calcule **toutes** les lignes de $Q$ quand $P = A$ est fixée (ou toutes les lignes de $P$ quand on lui passe $Q$ et la transposée de $E$). Réécrivez `moindres_carres_alternes` avec elle, et vérifiez que vous obtenez exactement le même résultat.

In [ ]:
def demi_etape(A, E, connu, lam):
    # À compléter : renvoie la matrice des lignes optimales, une par colonne de E
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def demi_etape(A, E, connu, lam):
    k = A.shape[1]
    resultat = np.zeros((E.shape[1], k))
    for f in range(E.shape[1]):
        i = connu[:, f]
        resultat[f] = np.linalg.solve(A[i].T @ A[i] + lam * np.eye(k), A[i].T @ E[i, f])
    return resultat

def moindres_carres_alternes_v2(E, connu, k=10, lam=10.0, iterations=15, graine=0):
    rng = np.random.default_rng(graine)
    P = rng.normal(0, 0.1, (E.shape[0], k))
    Q = rng.normal(0, 0.1, (E.shape[1], k))
    for _ in range(iterations):
        P = demi_etape(Q, E.T, connu.T, lam)
        Q = demi_etape(P, E, connu, lam)
    return P, Q

Pa, Qa = moindres_carres_alternes(E, connu, k=5, lam=10, iterations=5)
Pb, Qb = moindres_carres_alternes_v2(E, connu, k=5, lam=10, iterations=5)
print("Même résultat ?", np.allclose(Pa @ Qa.T, Pb @ Qb.T))

for graine in (1, 2):
    Pg, Qg = moindres_carres_alternes(E, connu, k=k_opt, lam=lam_opt, iterations=15, graine=graine)
    print(f"graine {graine} : REQM {evaluer(Pg @ Qg.T):.4f}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Combiner des outils pour produire le résultat": 1,
    "Transformer des constantes en paramètres": 1,
    "Décider si les résultats suffisent pour avancer ou abandonner": 1,
    "Comparer les mérites des outils dans le contexte": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "A3-facteurs-latents", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")